## 步驟 1：環境設置

安裝所需的 Python 套件

In [ ]:
# 安裝必要套件
!pip install datasets sentence-transformers transformers torch scikit-learn matplotlib pandas tqdm openai qiskit -q

print("✓ 套件安裝完成")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 3.5 MB/s eta 0:00:00
✓ 套件安裝完成


## 步驟 2：載入資料集

我們使用 `lopentu/Chinese-Wordnet-SemCor` 資料集，這是專門針對中文多義詞設計的 WSD 資料集。

In [ ]:
from datasets import load_dataset
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print("正在載入 Chinese-Wordnet-SemCor 資料集...")
ds = load_dataset('lopentu/Chinese-Wordnet-SemCor')
print(f"✓ 資料集載入完成")
print(f"\n訓練集大小: {len(ds['train'])} 筆")

# 查看資料結構
example = ds['train'][0]
print("\n資料欄位：")
for k, v in example.items():
    print(f"  {k}: {v}")

正在載入 Chinese-Wordnet-SemCor 資料集...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/22.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/7.51M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/284060 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/95410 [00:00<?, ? examples/s]

✓ 資料集載入完成

訓練集大小: 284060 筆

資料欄位：
  test_word: 死
  test_pos: VH
  test_sense_id: 05035202
  test_definition: 形容生物失去生命。
  test_sentence: 還不如<死>了算了。
  cwn_sense_id: 05035201
  cwn_definition: 形容沒有生命的。
  cwn_sentence: 噴過殺蟲劑，以入口處<死>蟑螂陳屍最多。
  label: False


### 📊 資料說明

每筆資料包含：
- `test_word`: 要消歧的目標詞
- `test_sentence`: 包含目標詞的句子（上下文）
- `cwn_definition`: 候選詞義的定義
- `label`: True 表示這是正確的詞義配對

## 步驟 3：準備測試資料

選擇幾個常見的多義詞進行測試

In [ ]:
# 選擇要測試的多義詞
test_words = ['死', '打', '開']  # 可以根據需要調整

# 為每個詞準備測試資料
test_data = {}
for word in test_words:
    examples = ds['train'].filter(lambda x: x['test_word'] == word)
    # 每個詞取 50 筆進行演示
    if len(examples) > 50:
        examples = examples.select(range(50))
    test_data[word] = examples
    print(f"'{word}': {len(examples)} 筆配對資料")




Filter:   0%|          | 0/284060 [00:00<?, ? examples/s]

'死': 50 筆配對資料


Filter:   0%|          | 0/284060 [00:00<?, ? examples/s]

'打': 50 筆配對資料


Filter:   0%|          | 0/284060 [00:00<?, ? examples/s]

'開': 50 筆配對資料


## 步驟 4：載入 BGE-large 模型

**BGE（BAAI General Embedding）** 是北京智源研究院開發的中文語義嵌入模型，在多項中文 NLP 任務中表現優異。

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

print("正在載入 BGE-large-zh-v1.5 模型...")
print("（首次載入可能需要幾分鐘下載模型檔案）\n")

bge_model = SentenceTransformer('BAAI/bge-large-zh-v1.5')

print("✓ BGE-large 模型載入完成")
print(f"模型維度: {bge_model.get_sentence_embedding_dimension()}")

正在載入 BGE-large-zh-v1.5 模型...
（首次載入可能需要幾分鐘下載模型檔案）



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.30G [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✓ BGE-large 模型載入完成
模型維度: 1024


## 步驟 5：定義雙向目標詞編碼方法

### 💡 核心概念

**雙向目標詞方法（Both-Target Method）**：
- 不使用整句的平均 embedding
- 而是提取目標詞周圍的**局部上下文**
- 對句子和定義都採用這個方法
- 這樣能更精準地捕捉目標詞在特定語境下的語義

In [ ]:
def encode_target_context(text, target_word, model, context_window=10):
    """
    提取目標詞周圍的上下文進行編碼

    參數:
        text: 輸入文本
        target_word: 目標詞
        model: 語義模型
        context_window: 目標詞前後保留的字元數

    返回:
        numpy array: 上下文的語義向量
    """
    idx = text.find(target_word)
    # print(idx)
    if idx != -1:
        # 提取目標詞前後各 context_window 個字元
        start = max(0, idx - context_window)
        end = min(len(text), idx + len(target_word) + context_window)
        context = text[start:end]
        return model.encode([context], convert_to_numpy=True)
    else:
        # 如果找不到目標詞，使用整句
        return model.encode([text], convert_to_numpy=True)

# 測試編碼函數
test_sentence = "他死了，但他的精神永遠活在我們心中"
test_word = "死"

embedding = encode_target_context(test_sentence, test_word, bge_model)
print(f"✓ 編碼函數測試成功")
print(f"輸入: {test_sentence}")
print(f"目標詞: {test_word}")
print(f"向量維度: {embedding.shape}")

✓ 編碼函數測試成功
輸入: 他死了，但他的精神永遠活在我們心中
目標詞: 死
向量維度: (1, 1024)


In [ ]:
# ======================================================
# [插入 Cell 1] 定義量子啟發式匹配系統
# 請確保已安裝 qiskit: !pip install qiskit -q
# ======================================================

import numpy as np
from qiskit.quantum_info import Statevector, DensityMatrix

class QuantumInspiredMatchingSystem:
    def __init__(self, embedding_dim):
        """
        初始化量子系統
        :param embedding_dim: 經典詞向量的維度 (例如 BGE-large 為 1024)
        """
        self.raw_dim = embedding_dim
        # 計算所需的 Qubits 數量 n，使得 2^n >= embedding_dim
        # 對於 1024 維，剛好需要 10 個 Qubits (2^10 = 1024)
        self.n_qubits = int(np.ceil(np.log2(embedding_dim)))
        self.quantum_dim = 2 ** self.n_qubits

        print(f"🌌 量子系統初始化完成:")
        print(f"   - 原始特徵維度: {self.raw_dim}")
        print(f"   - 映射希爾伯特空間維度: {self.quantum_dim} (使用 {self.n_qubits} Qubits)")

    def encode_to_quantum_state(self, amplitude_vec):
        """
        將經典向量編碼為量子純態 (Pure State) |ψ⟩
        """
        # 展平並確保是 numpy array
        amplitude_vec = np.array(amplitude_vec).flatten()

        # 1. 補零 (Padding): 如果維度不足 2^n，補 0
        if len(amplitude_vec) < self.quantum_dim:
            padding = np.zeros(self.quantum_dim - len(amplitude_vec))
            full_vec = np.concatenate([amplitude_vec, padding])
        else:
            full_vec = amplitude_vec[:self.quantum_dim] # 截斷以防萬一

        # 2. 歸一化 (L2 Normalization): 確保 <ψ|ψ> = 1
        norm = np.linalg.norm(full_vec)
        if norm > 1e-10:
            full_vec = full_vec / norm
        else:
            # 處理全零向量的情況，避免錯誤
            full_vec[0] = 1.0

        return Statevector(full_vec)

    def measure_similarity(self, sentence_vec, definition_vec):
        """
        執行量子測量 (Quantum Measurement)
        計算 Trace(ρ_sentence * |φ_def⟩⟨φ_def|)
        """
        # 步驟 A: 將兩個向量分別編碼為量子態
        psi_sentence = self.encode_to_quantum_state(sentence_vec)
        phi_definition = self.encode_to_quantum_state(definition_vec)

        # 步驟 B: 將句子狀態轉為密度矩陣 ρ (這裡暫用純態近似)
        # 論文核心概念：將語義封裝在密度矩陣中
        rho_sentence = DensityMatrix(psi_sentence)

        # 步驟 C: 使用定義態作為測量算符進行測量
        # 根據 Born Rule，這計算的是句子態塌縮到定義態的機率 (Fidelity)
        # 數學上等同於 |<psi|phi>|^2
        probability = rho_sentence.expectation_value(phi_definition).real

        return probability

# 初始化系統 (BGE-large-zh 的維度是 1024)
q_system = QuantumInspiredMatchingSystem(embedding_dim=1024)

🌌 量子系統初始化完成:
   - 原始特徵維度: 1024
   - 映射希爾伯特空間維度: 1024 (使用 10 Qubits)


In [ ]:
# ======================================================
# [插入 Cell 2] 定義並測試量子版 WSD 預測函數
# ======================================================

def wsd_predict_quantum(sentence, target_word, candidate_definitions, model, q_system):
    """
    使用量子測量機率進行詞義消歧
    """
    # 1. 取得句子的 Context Embedding (利用 Notebook 前面定義好的 encode_target_context)
    # 注意：取 [0] 是因為 encode 返回的是 (1, 1024) 的矩陣
    sentence_emb = encode_target_context(sentence, target_word, model)[0]

    scores = []

    # 2. 遍歷所有候選定義
    for definition, _ in candidate_definitions:
        # 取得定義的 Embedding
        def_emb = encode_target_context(definition, target_word, model)[0]

        # 3. 【核心差異】呼叫量子系統計算「測量機率」而非餘弦相似度
        prob = q_system.measure_similarity(sentence_emb, def_emb)
        scores.append(prob)

    # 4. 找出機率最高的索引
    best_idx = np.argmax(scores)

    return best_idx, scores

# --- 測試案例 (驗證整合是否成功) ---
print("\n🔍 --- 執行量子版 WSD 測試 ---")
test_sent_q = "這個應用程式的視窗被我不小心關掉了"
target_word_q = "視窗"
test_defs_q = [
    ("牆上用來採光通風的洞口", False),  # 物理窗戶
    ("電腦螢幕上顯示程式執行內容的矩形區域", True) # 電腦視窗
]

# 執行預測
pred_idx, q_scores = wsd_predict_quantum(test_sent_q, target_word_q, test_defs_q, bge_model, q_system)

print(f"句子: {test_sent_q}")
print(f"目標詞: {target_word_q}")
print("-" * 30)
for i, ((defi, _), score) in enumerate(zip(test_defs_q, q_scores)):
    marker = "✅ 預測選取" if i == pred_idx else "   "
    print(f"{marker} | 定義: {defi[:15]}... | 量子匹配機率: {score:.6f}")


🔍 --- 執行量子版 WSD 測試 ---
句子: 這個應用程式的視窗被我不小心關掉了
目標詞: 視窗
------------------------------
    | 定義: 牆上用來採光通風的洞口... | 量子匹配機率: 0.165364
✅ 預測選取 | 定義: 電腦螢幕上顯示程式執行內容的矩... | 量子匹配機率: 0.250589


In [ ]:
# ======================================================
# [新增 Cell] Word2State 混合態 (Mixed State) 量子啟發匹配系統
# ======================================================

import numpy as np
from qiskit.quantum_info import Statevector, DensityMatrix
from qiskit.quantum_info import Operator

class Word2StateMixedMatchingSystem:
    def __init__(self, embedding_dim: int):
        self.raw_dim = embedding_dim
        self.n_qubits = int(np.ceil(np.log2(embedding_dim)))
        self.quantum_dim = 2 ** self.n_qubits
        print(
            f"🌌 Word2State(混合態) 初始化完成: "
            f"raw_dim={self.raw_dim}, quantum_dim={self.quantum_dim}, qubits={self.n_qubits}"
        )

    def encode_to_pure_state(self, vec):
        vec = np.array(vec).flatten()

        # pad / truncate to 2^n
        if len(vec) < self.quantum_dim:
            vec = np.concatenate([vec, np.zeros(self.quantum_dim - len(vec))])
        else:
            vec = vec[:self.quantum_dim]

        # normalize
        norm = np.linalg.norm(vec)
        if norm > 1e-10:
            vec = vec / norm
        else:
            vec = np.zeros(self.quantum_dim)
            vec[0] = 1.0

        return Statevector(vec)

    def build_mixed_density(self, component_vectors, weights=None):
        # ✅ 注意：整個 function body 都要縮排 4 個空白
        comps = np.array(component_vectors)
        m = comps.shape[0]

        # weights c_j
        if weights is None:
            c = np.ones(m) / m
        else:
            c = np.array(weights).flatten()
            c = np.clip(c, 0.0, None)
            s = c.sum()
            c = c / s if s > 1e-12 else (np.ones(m) / m)

        rho = None
        for j in range(m):
            psi_j = self.encode_to_pure_state(comps[j])
            rho_j = DensityMatrix(psi_j)     # |psi_j><psi_j|
            rho_j = rho_j * float(c[j])      # ✅ 正確的 scalar multiplication

            rho = rho_j if rho is None else (rho + rho_j)

        return rho

    def measure_probability(self, sentence_components, definition_vec, weights=None):
        rho = self.build_mixed_density(sentence_components, weights=weights)
        phi = self.encode_to_pure_state(definition_vec)

        projector = Operator(phi.to_operator().data)
        p = rho.expectation_value(projector)
        return float(np.real(p))


# ======================================================
# components 產生（Word2State：context → 多個 components）
# ======================================================

def encode_context_components(
    text,
    target_word,
    model,
    context_window=10,
    chunk_chars=8,
    stride=4
):
    """
    取目標詞附近 context，切成多個小片段，每個片段各自做 embedding。
    回傳:
      comps: (m, dim)
      chunks: list[str]
    """
    idx = text.find(target_word)
    if idx != -1:
        start = max(0, idx - context_window)
        end = min(len(text), idx + len(target_word) + context_window)
        context = text[start:end]
    else:
        context = text

    chunks = []
    for s in range(0, max(1, len(context) - chunk_chars + 1), stride):
        chunk = context[s:s + chunk_chars].strip()
        if chunk:
            chunks.append(chunk)

    if not chunks:
        chunks = [context]

    comps = model.encode(chunks, convert_to_numpy=True)  # (m, dim)
    return comps, chunks


# ======================================================
# 初始化 Word2State 混合態系統
# ======================================================

q_system_mixed = Word2StateMixedMatchingSystem(
    embedding_dim=bge_model.get_sentence_embedding_dimension()
)


🌌 Word2State(混合態) 初始化完成: raw_dim=1024, quantum_dim=1024, qubits=10


In [ ]:
# ======================================================
# [新增 Cell] Word2State 混合態量子版 WSD 預測（可直接跑）
# 依賴：
#   - bge_model 已載入
#   - encode_context_components 已定義
#   - q_system_mixed = Word2StateMixedMatchingSystem(...) 已初始化
#   - q_system_mixed.measure_probability(...) 內部 build_mixed_density 已修正（用 * scalar）
# ======================================================

import numpy as np
# --- 安全檢查：避免你少跑某個 cell ---
assert "bge_model" in globals(), "❌ 找不到 bge_model：請先執行載入 BGE 的 Cell"
assert "encode_context_components" in globals(), "❌ 找不到 encode_context_components：請先執行 components 產生的 Cell"
assert "q_system_mixed" in globals(), "❌ 找不到 q_system_mixed：請先初始化 Word2StateMixedMatchingSystem"

def wsd_predict_quantum_mixed(
    sentence,
    target_word,
    candidate_definitions,
    model,
    q_system,
    context_window=10,
    chunk_chars=8,
    stride=4,
    weights=None,
    debug=False
):
    """
    用 Word2State 混合態量子測量機率做 WSD

    參數:
      sentence: 句子 (str)
      target_word: 多義詞 (str)
      candidate_definitions: [(definition_text, label/anything), ...]
      model: SentenceTransformer
      q_system: Word2StateMixedMatchingSystem
      weights: None(均勻) 或 (m,) 權重
      debug: True 時印出切片 chunks

    回傳:
      best_idx: 分數最高的定義索引
      scores: 每個定義的量子匹配機率
      sent_chunks: 用來混合的 context chunks（方便你看切了哪些）
    """
    # 1) sentence -> components (m, dim)
    sent_comps, sent_chunks = encode_context_components(
        sentence, target_word, model,
        context_window=context_window,
        chunk_chars=chunk_chars,
        stride=stride
    )

    if debug:
        print("DEBUG chunks:", sent_chunks)
        print("DEBUG sent_comps shape:", sent_comps.shape)

    scores = []
    # 2) definitions -> embedding -> measurement probability
    for definition, _ in candidate_definitions:
        # ✅ 對 definition 不用 encode_target_context（避免 target_word 找不到造成策略不一致）
        def_emb = model.encode([definition], convert_to_numpy=True)[0]

        # p = Tr(rho_sentence * |phi><phi|)
        p = q_system.measure_probability(sent_comps, def_emb, weights=weights)
        scores.append(float(p))

    best_idx = int(np.argmax(scores))
    return best_idx, scores, sent_chunks


# ------------------ 測試案例 ------------------
print("\n🔍 --- 執行 Word2State(混合態) 量子版 WSD 測試 ---")
test_sent_q = "這個應用程式的視窗被我不小心關掉了"
target_word_q = "視窗"
test_defs_q = [
    ("牆上用來採光通風的洞口", False),
    ("電腦螢幕上顯示程式執行內容的矩形區域", True),
]

pred_idx, q_scores, used_chunks = wsd_predict_quantum_mixed(
    test_sent_q,
    target_word_q,
    test_defs_q,
    bge_model,
    q_system_mixed,
    context_window=10,
    chunk_chars=8,
    stride=4,
    weights=None,
    debug=True   # 你可以改 False
)

print(f"句子: {test_sent_q}")
print(f"目標詞: {target_word_q}")
print(f"（sentence 被切成的 components/chunks）: {used_chunks}")
print("-" * 30)
for i, ((defi, _), score) in enumerate(zip(test_defs_q, q_scores)):
    marker = "✅ 預測選取" if i == pred_idx else "   "
    print(f"{marker} | 定義: {defi[:15]}... | 量子匹配機率: {score:.6f}")



🔍 --- 執行 Word2State(混合態) 量子版 WSD 測試 ---
DEBUG chunks: ['這個應用程式的視', '程式的視窗被我不', '窗被我不小心關掉']
DEBUG sent_comps shape: (3, 1024)
句子: 這個應用程式的視窗被我不小心關掉了
目標詞: 視窗
（sentence 被切成的 components/chunks）: ['這個應用程式的視', '程式的視窗被我不', '窗被我不小心關掉']
------------------------------
    | 定義: 牆上用來採光通風的洞口... | 量子匹配機率: 0.181480
✅ 預測選取 | 定義: 電腦螢幕上顯示程式執行內容的矩... | 量子匹配機率: 0.220380


In [ ]:
# ======================================================
# [新增 Cell] Word2State 混合態（definition-dependent weights）
# ======================================================

import numpy as np

def predict_word2state_mixed_weighted(
    ex,
    model,
    q_system_mixed,
    context_window=10,
    chunk_chars=12,
    stride=6
):
    """
    Word2State mixed with definition-dependent weights:
    c_j ∝ max(0, cosine(chunk_j, definition))
    """
    sent = ex["sentence"]
    word = ex["target_word"]
    defs = ex["definitions"]

    # sentence -> components
    sent_comps, _ = encode_context_components(
        sent,
        word,
        model,
        context_window=context_window,
        chunk_chars=chunk_chars,
        stride=stride
    )

    comp_norms = np.linalg.norm(sent_comps, axis=1) + 1e-12

    scores = []
    for d_text, _ in defs:
        def_emb = model.encode([d_text], convert_to_numpy=True)[0]
        def_norm = np.linalg.norm(def_emb) + 1e-12

        # 🔑 關鍵改動：definition-dependent weights
        cos_vec = (sent_comps @ def_emb) / (comp_norms * def_norm)
        weights = np.maximum(0.0, cos_vec)   # ReLU

        # fallback：如果全為 0，用均勻
        if weights.sum() < 1e-12:
            weights = None

        p = q_system_mixed.measure_probability(
            sent_comps,
            def_emb,
            weights=weights
        )
        scores.append(float(p))

    pred_idx = int(np.argmax(scores))
    return pred_idx, scores


## 步驟 6：實作 WSD 系統

核心演算法：
1. 對測試句子提取目標詞的上下文 embedding
2. 對每個候選定義提取目標詞的上下文 embedding
3. 計算餘弦相似度
4. 選擇相似度最高的定義

In [ ]:
def wsd_predict(sentence, target_word, candidate_definitions, model):
    """
    詞義消歧預測函數

    參數:
        sentence: 包含目標詞的句子
        target_word: 要消歧的詞
        candidate_definitions: 候選定義列表 [(定義文本, 是否正確), ...]
        model: 語義模型

    返回:
        best_idx: 預測的最佳定義索引
        scores: 所有候選定義的相似度分數
    """
    # 編碼句子中的目標詞上下文
    sentence_emb = encode_target_context(sentence, target_word, model)

    scores = []
    for definition, _ in candidate_definitions:
        # 編碼定義中的目標詞上下文
        def_emb = encode_target_context(definition, target_word, model)
        # 計算餘弦相似度
        similarity = cosine_similarity(sentence_emb, def_emb)[0][0]
        scores.append(similarity)

    best_idx = np.argmax(scores)
    return best_idx, scores

# 測試預測函數
test_sentence = "這個機器已經死了，無法啟動"
test_definitions = [
    ("生命終止，失去生命", False),
    ("機器或設備停止運作，無法使用", True),
    ("固定不動，沒有彈性", False)
]

pred_idx, scores = wsd_predict(test_sentence, "死", test_definitions, bge_model)

print("測試案例：")
print(f"句子: {test_sentence}\n")
print("候選定義與分數:")
for i, ((definition, is_correct), score) in enumerate(zip(test_definitions, scores)):
    marker = "✓" if is_correct else " "
    predict_marker = "👉" if i == pred_idx else "  "
    print(f"{predict_marker} [{marker}] {score:.4f} - {definition}")

correct = test_definitions[pred_idx][1]
print(f"\n預測結果: {'✓ 正確' if correct else '✗ 錯誤'}")

測試案例：
句子: 這個機器已經死了，無法啟動

候選定義與分數:
   [ ] 0.5166 - 生命終止，失去生命
👉 [✓] 0.7322 - 機器或設備停止運作，無法使用
   [ ] 0.5635 - 固定不動，沒有彈性

預測結果: ✓ 正確


## 步驟 7：在測試集上評估 BGE-large

對所有測試詞進行完整評估

In [ ]:
def evaluate_wsd(examples, target_word, model, verbose=True):
    """
    評估 WSD 系統的準確率
    """
    # 按句子分組（每個句子可能有多個候選定義）
    sentence_groups = {}
    for i in range(len(examples)):
        e = examples[i]
        sent = e['test_sentence']
        if sent not in sentence_groups:
            sentence_groups[sent] = []
        sentence_groups[sent].append((e['cwn_definition'], e['label']))

    correct_count = 0
    total_count = len(sentence_groups)

    if verbose:
        print(f"評估 '{target_word}' 的 {total_count} 個句子...\n")

    # 儲存一些案例用於展示
    examples_to_show = []

    for sent, candidates in sentence_groups.items():
        pred_idx, scores = wsd_predict(sent, target_word, candidates, model)

        # 檢查預測是否正確
        is_correct = candidates[pred_idx][1]
        if is_correct:
            correct_count += 1

        # 保存前 3 個案例（包含正確和錯誤的）
        if len(examples_to_show) < 3:
            examples_to_show.append({
                'sentence': sent,
                'candidates': candidates,
                'scores': scores,
                'predicted': pred_idx,
                'correct': is_correct
            })

    accuracy = correct_count / total_count

    if verbose:
        print(f"\n{'='*70}")
        print(f"準確率: {accuracy:.2%} ({correct_count}/{total_count})")
        print(f"{'='*70}\n")

        # 顯示案例
        print("案例展示：\n")
        for i, example in enumerate(examples_to_show, 1):
            print(f"案例 {i}: {'✓ 正確' if example['correct'] else '✗ 錯誤'}")
            print(f"句子: {example['sentence']}")
            print(f"候選定義:")
            for j, ((definition, is_true), score) in enumerate(zip(example['candidates'], example['scores'])):
                marker = "[正解]" if is_true else "      "
                predict_marker = "👉" if j == example['predicted'] else "  "
                print(f"  {predict_marker} {marker} {score:.4f} - {definition}")
            print()

    return accuracy, correct_count, total_count

# 對所有測試詞進行評估
print("\n" + "="*70)
print("BGE-large + 雙向目標詞方法 評估結果")
print("="*70 + "\n")

bge_results = {}
for word in test_words:
    print(f"\n{'='*70}")
    print(f"測試詞: '{word}'")
    print(f"{'='*70}")

    accuracy, correct, total = evaluate_wsd(test_data[word], word, bge_model, verbose=True)
    bge_results[word] = accuracy

# 總結
avg_accuracy = np.mean(list(bge_results.values()))
print(f"\n{'='*70}")
print(f"🎯 平均準確率: {avg_accuracy:.2%}")
print(f"{'='*70}")


BGE-large + 雙向目標詞方法 評估結果


測試詞: '死'
評估 '死' 的 5 個句子...


準確率: 20.00% (1/5)

案例展示：

案例 1: ✗ 錯誤
句子: 還不如<死>了算了。
候選定義:
            0.5508 - 形容沒有生命的。
            0.5131 - 形容比喻事件終止不再存在。
  👉        0.6239 - 使後述對象失去生命。
            0.5546 - 形容比喻沒有生命的。
            0.4217 - 形容機器的特定功能喪失。
            0.4962 - 形容比喻不能活動的。
            0.4253 - 形容不能流動的。
            0.4702 - 形容不再使用的。
            0.4160 - 形容比喻沒變化的或不可改變的。
            0.4734 - 形容在競爭中被淘汰。
            0.4051 - 形容指數下跌。
     [正解] 0.5179 - 形容生物失去生命。

案例 2: ✗ 錯誤
句子: <死>都不怕，
候選定義:
  👉        0.4847 - 形容沒有生命的。
            0.3704 - 形容比喻事件終止不再存在。
            0.4527 - 使後述對象失去生命。
            0.4739 - 形容比喻沒有生命的。
            0.2901 - 形容機器的特定功能喪失。
            0.4174 - 形容比喻不能活動的。
            0.3229 - 形容不能流動的。
            0.3796 - 形容不再使用的。
            0.4113 - 形容比喻沒變化的或不可改變的。
            0.3266 - 形容在競爭中被淘汰。
            0.2907 - 形容指數下跌。
     [正解] 0.4101 - 形容生物失去生命。

案例 3: ✗ 錯誤
句子: 就讓你<死>得心裡明白。
候選定義:
            0.5232 - 形容沒有生命的。
            0.4934 - 形容比喻事件終

## 步驟 8：深入分析 Embedding 特徵

讓我們看看 BGE 模型生成的 embedding 向量的數值特性和格式

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 選擇一個測試案例
test_word = '死'
test_sentence = test_data[test_word][0]['test_sentence']
test_definitions = [test_data[test_word][i]['cwn_definition'] for i in range(min(3, len(test_data[test_word])))]

print("="*80)
print("Embedding 特徵分析")
print("="*80)
print(f"\n目標詞: '{test_word}'")
print(f"測試句子: {test_sentence}\n")

# 生成 embedding
sent_embedding = encode_target_context(test_sentence, test_word, bge_model)

print("\n【1. Embedding 基本資訊】")
print(f"向量維度: {sent_embedding.shape}")
print(f"向量長度: {sent_embedding.shape[1]} 維")
print(f"資料型態: {sent_embedding.dtype}")
print(f"向量範數 (L2 norm): {np.linalg.norm(sent_embedding):.4f}")

print("\n【2. Embedding 數值統計】")
print(f"最小值: {sent_embedding.min():.6f}")
print(f"最大值: {sent_embedding.max():.6f}")
print(f"平均值: {sent_embedding.mean():.6f}")
print(f"標準差: {sent_embedding.std():.6f}")
print(f"中位數: {np.median(sent_embedding):.6f}")

print("\n【3. Embedding 向量片段（前 20 維）】")
print("索引 | 數值")
print("-" * 30)
for i in range(min(20, sent_embedding.shape[1])):
    print(f"{i:4d} | {sent_embedding[0, i]:+.6f}")

print("\n【4. 數值分佈統計】")
positive_count = np.sum(sent_embedding > 0)
negative_count = np.sum(sent_embedding < 0)
zero_count = np.sum(sent_embedding == 0)
print(f"正數: {positive_count} ({positive_count/sent_embedding.size*100:.1f}%)")
print(f"負數: {negative_count} ({negative_count/sent_embedding.size*100:.1f}%)")
print(f"零值: {zero_count} ({zero_count/sent_embedding.size*100:.1f}%)")

print("\n【5. 不同文本的 Embedding 比較】")
print("\n比較句子和定義的 embedding:")
for i, definition in enumerate(test_definitions[:3], 1):
    def_embedding = encode_target_context(definition, test_word, bge_model)
    similarity = cosine_similarity(sent_embedding, def_embedding)[0][0]

    print(f"\n定義 {i}: {definition}")
    print(f"  向量範數: {np.linalg.norm(def_embedding):.4f}")
    print(f"  平均值: {def_embedding.mean():.6f}")
    print(f"  標準差: {def_embedding.std():.6f}")
    print(f"  與句子的餘弦相似度: {similarity:.4f}")

Embedding 特徵分析

目標詞: '死'
測試句子: 還不如<死>了算了。


【1. Embedding 基本資訊】
向量維度: (1, 1024)
向量長度: 1024 維
資料型態: float32
向量範數 (L2 norm): 1.0000

【2. Embedding 數值統計】
最小值: -0.084482
最大值: 0.386263
平均值: -0.000767
標準差: 0.031241
中位數: -0.002441

【3. Embedding 向量片段（前 20 維）】
索引 | 數值
------------------------------
   0 | -0.013173
   1 | +0.016819
   2 | +0.036227
   3 | -0.052376
   4 | +0.009854
   5 | -0.022115
   6 | -0.036511
   7 | +0.017109
   8 | +0.009383
   9 | -0.026118
  10 | +0.001029
  11 | -0.003689
  12 | +0.032458
  13 | -0.017257
  14 | -0.036240
  15 | +0.008203
  16 | +0.014860
  17 | +0.040044
  18 | -0.027202
  19 | -0.006108

【4. 數值分佈統計】
正數: 480 (46.9%)
負數: 544 (53.1%)
零值: 0 (0.0%)

【5. 不同文本的 Embedding 比較】

比較句子和定義的 embedding:

定義 1: 形容沒有生命的。
  向量範數: 1.0000
  平均值: -0.000707
  標準差: 0.031242
  與句子的餘弦相似度: 0.5508

定義 2: 形容比喻事件終止不再存在。
  向量範數: 1.0000
  平均值: -0.000878
  標準差: 0.031238
  與句子的餘弦相似度: 0.5131

定義 3: 使後述對象失去生命。
  向量範數: 1.0000
  平均值: -0.000809
  標準差: 0.031240
  與句子的餘弦相似度: 0.6239


In [ ]:
# =============================================================================
# [插入此 Cell 於 評估函數 之前]：製作 test_dataset
# =============================================================================

# 初始化測試集列表
test_dataset = []

print("正在重新整理資料格式...")

# 遍歷我們之前選出的測試詞 (死, 打, 開)
for word, hf_dataset in test_data.items():
    # 使用字典來將同一句子的不同候選定義收集在一起
    # key: 句子, value: 候選定義列表
    sentence_groups = {}

    for example in hf_dataset:
        sent = example['test_sentence']
        if sent not in sentence_groups:
            sentence_groups[sent] = []
        # 收集定義與標籤: (定義內容, 是否正確)
        sentence_groups[sent].append((example['cwn_definition'], example['label']))

    # 將整理好的分組資料加入 test_dataset
    for sent, definitions in sentence_groups.items():
        test_dataset.append({
            'sentence': sent,
            'target_word': word,
            'definitions': definitions
        })

print(f"✓ 資料轉換完成！共整理出 {len(test_dataset)} 筆唯一的測試句子。")
print(f"  範例結構: {test_dataset[0].keys()}")

正在重新整理資料格式...
✓ 資料轉換完成！共整理出 7 筆唯一的測試句子。
  範例結構: dict_keys(['sentence', 'target_word', 'definitions'])


In [ ]:
# =============================================================================
# [插入此 Cell 於 步驟 7/8 後]：在測試集上評估量子方法的準確率
# =============================================================================

from tqdm.notebook import tqdm
def evaluate_quantum_wsd(dataset, model, q_system, limit=None):
    """
    在測試集上評估量子啟發式 WSD 的效能
    """
    correct_count = 0
    total_count = 0

    # 如果只想快速測試，可以設定 limit (例如 limit=100)
    data_to_eval = dataset[:limit] if limit else dataset

    print(f"🚀 開始量子啟發式 WSD 全局評估 (共 {len(data_to_eval)} 筆數據)...")

    # 使用 tqdm 顯示進度條
    for item in tqdm(data_to_eval, desc="Quantum Processing"):
        # 1. 解包數據 (請確保這裡的 key 對應您前面資料處理的欄位名稱)
        sentence = item['sentence']
        target_word = item['target_word']
        definitions = item['definitions'] # 預期格式: [(def_text, label_bool), ...]

        # 找出正確答案的 index (Gold Label)
        # 假設 definitions 列表結構是 [("定義內容", True/False), ...]
        ground_truth_idx = -1
        for idx, (_, is_correct) in enumerate(definitions):
            if is_correct:
                ground_truth_idx = idx
                break

        if ground_truth_idx == -1: continue # 跳過沒有標準答案的數據

        # 2. 執行量子預測
        pred_idx, scores = wsd_predict_quantum(sentence, target_word, definitions, model, q_system)

        # 3. 比對結果
        if pred_idx == ground_truth_idx:
            correct_count += 1

        total_count += 1

    # 4. 輸出統計結果
    accuracy = correct_count / total_count if total_count > 0 else 0
    print(f"\n📊 量子方法評估報告 (Quantum-Inspired Method):")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    print(f"   ● 測試樣本數 : {total_count}")
    print(f"   ● 預測正確數 : {correct_count}")
    print(f"   ● 總體準確率 : {accuracy:.2%}")
    print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")

    return accuracy

# --- 執行評估 ---
# 請確認您的測試集變數名稱 (通常是 test_dataset 或 test_data)
# 這裡設定 limit=200 先跑前 200 筆看看速度與效果，若速度可接受可移除 limit
if 'test_dataset' in globals():
    acc = evaluate_quantum_wsd(test_dataset, bge_model, q_system, limit=200)
else:
    print("⚠️ 找不到 test_dataset 變數，請確認您前面的資料集變數名稱 (例如 dataset, val_data 等)")

🚀 開始量子啟發式 WSD 全局評估 (共 7 筆數據)...


Quantum Processing:   0%|          | 0/7 [00:00<?, ?it/s]


📊 量子方法評估報告 (Quantum-Inspired Method):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   ● 測試樣本數 : 6
   ● 預測正確數 : 2
   ● 總體準確率 : 33.33%
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [ ]:
# ======================================================
# [新增 Cell] 建立 Word2State(混合態) 專用 test_dataset（依你的欄位）
# ======================================================

from collections import defaultdict

def build_word2state_test_dataset_from_pairwise(
    ds_split,
    max_instances=300,          # 控制總題數（可調大）
    max_candidates=None,        # 每題最多幾個候選（None = 不限）
    use_cwn_sentence=False,     # True 用 cwn_sentence；False 用 test_sentence
    use_cwn_def=False           # True 用 cwn_definition/cwn_sense_id；False 用 test_definition/test_sense_id
):
    # 選句子/定義來源
    sent_key = "cwn_sentence" if use_cwn_sentence else "test_sentence"
    def_key  = "cwn_definition" if use_cwn_def else "test_definition"
    sid_key  = "cwn_sense_id" if use_cwn_def else "test_sense_id"

    groups = defaultdict(list)

    # 以 (target_word, sentence) 分組，把同一句的候選 definition 收集起來
    for ex in ds_split:
        word = ex["test_word"]
        sent = ex[sent_key]
        groups[(word, sent)].append(ex)

    test_dataset = []
    skipped_no_gold = 0
    skipped_too_few = 0

    for (word, sent), rows in groups.items():
        # 收集候選（避免重複 sense）
        seen = set()
        definitions = []
        gold_index = None

        for r in rows:
            sid = r[sid_key]
            defi = r[def_key]

            if sid in seen:
                continue
            seen.add(sid)

            definitions.append((defi, sid))

        # 找 gold（label==1 那個 sense）
        gold_rows = [r for r in rows if int(r["label"]) == 1]
        if len(gold_rows) == 0:
            skipped_no_gold += 1
            continue

        gold_sid = gold_rows[0][sid_key]
        for i, (_, sid) in enumerate(definitions):
            if sid == gold_sid:
                gold_index = i
                break

        if gold_index is None:
            skipped_no_gold += 1
            continue

        # 至少要有 2 個候選才是 WSD
        if len(definitions) < 2:
            skipped_too_few += 1
            continue

        # 可選：限制候選數（保留 gold + 前幾個）
        if max_candidates is not None and len(definitions) > max_candidates:
            # 確保 gold 留下來
            gold_def = definitions[gold_index]
            others = [d for i, d in enumerate(definitions) if i != gold_index]
            others = others[:max_candidates - 1]
            definitions = [gold_def] + others
            gold_index = 0

        test_dataset.append({
            "sentence": sent,
            "target_word": word,
            "definitions": definitions,
            "gold_index": gold_index
        })

        if len(test_dataset) >= max_instances:
            break

    print("✓ test_dataset 建立完成")
    print("總題數:", len(test_dataset))
    print("略過（無 gold）:", skipped_no_gold)
    print("略過（候選 <2）:", skipped_too_few)
    print("使用句子欄位:", sent_key)
    print("使用定義欄位:", def_key, "/", sid_key)

    return test_dataset


# ✅ 建立（先做 300 題跑看看）
test_dataset = build_word2state_test_dataset_from_pairwise(
    ds["train"],
    max_instances=300,
    max_candidates=5,        # 你可以先限制到 5，速度比較快
    use_cwn_sentence=False,
    use_cwn_def=False
)

# 看一筆
ex = test_dataset[0]
print("\n--- 範例 ---")
print("sentence:", ex["sentence"])
print("target_word:", ex["target_word"])
print("gold_index:", ex["gold_index"])
print("candidates:", len(ex["definitions"]))
for i, (d, sid) in enumerate(ex["definitions"]):
    mark = "✅" if i == ex["gold_index"] else "  "
    print(f"{mark} [{i}] sid={sid} | {d[:30]}...")


✓ test_dataset 建立完成
總題數: 29
略過（無 gold）: 0
略過（候選 <2）: 21069
使用句子欄位: test_sentence
使用定義欄位: test_definition / test_sense_id

--- 範例 ---
sentence: <中>。
target_word: 中
gold_index: 0
candidates: 2
✅ [0] sid=04004617 | 年齡在四十到六十歲間。...
   [1] sid=08011201 | 國名，位於亞洲東部的國家，東瀕太平洋，北臨俄國，是世界上人口...


In [ ]:
correct = 0
for ex in test_dataset:
    pred_idx, _, _ = wsd_predict_quantum_mixed(
        ex["sentence"],
        ex["target_word"],
        ex["definitions"],
        bge_model,
        q_system_mixed
    )
    correct += (pred_idx == ex["gold_index"])

print("Word2State(混合態) accuracy =", correct / len(test_dataset))


Word2State(混合態) accuracy = 0.5172413793103449


In [ ]:
import numpy as np

# ---------- 1) Cosine baseline ----------
def predict_cosine(ex, model):
    sent = ex["sentence"]
    word = ex["target_word"]
    defs = ex["definitions"]

    sent_emb = encode_target_context(sent, word, model)[0]  # (1024,)

    scores = []
    for d_text, _ in defs:
        def_emb = model.encode([d_text], convert_to_numpy=True)[0]
        # cosine = dot(u,v)/(||u|| ||v||)
        denom = (np.linalg.norm(sent_emb) * np.linalg.norm(def_emb) + 1e-12)
        cos = float(np.dot(sent_emb, def_emb) / denom)
        scores.append(cos)

    pred_idx = int(np.argmax(scores))
    return pred_idx, scores

# ---------- 2) 量子純態 ----------
def predict_quantum_pure(ex, model, q_system_pure):
    """
    如果你純態系統 class 名不叫 q_system_pure.measure_similarity，
    你只要把下面那行改成你原本純態的呼叫即可。
    """
    sent = ex["sentence"]
    word = ex["target_word"]
    defs = ex["definitions"]

    sent_emb = encode_target_context(sent, word, model)[0]

    scores = []
    for d_text, _ in defs:
        def_emb = model.encode([d_text], convert_to_numpy=True)[0]
        p = q_system_pure.measure_similarity(sent_emb, def_emb)  # ← 純態版通常是這個
        scores.append(float(p))

    pred_idx = int(np.argmax(scores))
    return pred_idx, scores

# ---------- 3) Word2State 混合態 ----------
def predict_word2state_mixed(ex, model, q_system_mixed,
                             context_window=10, chunk_chars=8, stride=4):
    sent = ex["sentence"]
    word = ex["target_word"]
    defs = ex["definitions"]

    sent_comps, _ = encode_context_components(
        sent, word, model,
        context_window=context_window,
        chunk_chars=chunk_chars,
        stride=stride
    )

    scores = []
    for d_text, _ in defs:
        def_emb = model.encode([d_text], convert_to_numpy=True)[0]
        p = q_system_mixed.measure_probability(sent_comps, def_emb, weights=None)
        scores.append(float(p))

    pred_idx = int(np.argmax(scores))
    return pred_idx, scores

# ---------- 評估工具 ----------
def margin(scores):
    if len(scores) < 2:
        return 0.0
    s = sorted(scores, reverse=True)
    return float(s[0] - s[1])

def evaluate_three_methods(test_dataset, model, q_system_pure, q_system_mixed,
                           n_eval=200, seed=0, min_sentence_length=25):
    rng = np.random.default_rng(seed)

    # 過濾出符合句子長度條件的索引
    eligible_idxs = [i for i, ex in enumerate(test_dataset) if len(ex["sentence"]) > min_sentence_length]

    if len(eligible_idxs) == 0:
        print("警告：沒有足夠符合條件的句子進行評估。")
        return {
            "cosine": {"accuracy": 0.0, "avg_margin": 0.0},
            "pure":   {"accuracy": 0.0, "avg_margin": 0.0},
            "mixed":  {"accuracy": 0.0, "avg_margin": 0.0},
        }

    # 從符合條件的索引中隨機選擇 n_eval 個
    actual_n_eval = min(n_eval, len(eligible_idxs))
    selected_idxs = rng.choice(eligible_idxs, size=actual_n_eval, replace=False)

    results = {
        "cosine": {"correct": 0, "margins": []},
        "pure":   {"correct": 0, "margins": []},
        "mixed":  {"correct": 0, "margins": []},
    }

    for i in selected_idxs:
        ex = test_dataset[i]
        gold = ex["gold_index"]

        # cosine
        pred_c, sc_c = predict_cosine(ex, model)
        results["cosine"]["correct"] += int(pred_c == gold)
        results["cosine"]["margins"].append(margin(sc_c))

        # pure
        pred_p, sc_p = predict_quantum_pure(ex, model, q_system_pure)
        results["pure"]["correct"] += int(pred_p == gold)
        results["pure"]["margins"].append(margin(sc_p))

        # mixed
        pred_m, sc_m = predict_word2state_mixed_weighted(ex, model, q_system_mixed)
        results["mixed"]["correct"] += int(pred_m == gold)
        results["mixed"]["margins"].append(margin(sc_m))

    n = len(selected_idxs)
    summary = {
        k:
            {
            "accuracy": results[k]["correct"] / n if n > 0 else 0.0,
            "avg_margin": float(np.mean(results[k]["margins"])) if n > 0 else 0.0
        } for k in results
    }
    return summary

# ------------------ 你只要改這兩行 ------------------
# q_system_pure = 你原本的純態系統物件（QuantumInspiredMatchingSystem 那個）
# q_system_mixed = 你現在的 Word2StateMixedMatchingSystem 物件

print("開始比較三者（抽樣 200 題）...")
summary = evaluate_three_methods(
    test_dataset,
    bge_model,
    q_system_pure=q_system,        # ← 這裡改成你的純態系統變數名
    q_system_mixed=q_system_mixed,
    n_eval=200,
    seed=42,
    min_sentence_length=0 # 新增的參數
)

print("\n=== 結果（accuracy / avg_margin） ===")
for k, v in summary.items():
    print(f"{k:>6} | acc={v['accuracy']:.4f} | avg_margin={v['avg_margin']:.6f}")

開始比較三者（抽樣 200 題）...

=== 結果（accuracy / avg_margin） ===
cosine | acc=0.5517 | avg_margin=0.057964
  pure | acc=0.5517 | avg_margin=0.053154
 mixed | acc=0.5517 | avg_margin=0.050655


In [ ]:
# @title
# ======================================================
# [新增 Cell] 找 mixed vs cosine 分歧的題目（含分數、gold、chunks）
# ======================================================

import numpy as np

def predict_cosine_verbose(ex, model):
    sent = ex["sentence"]
    word = ex["target_word"]
    defs = ex["definitions"]

    sent_emb = encode_target_context(sent, word, model)[0]

    scores = []
    for d_text, _ in defs:
        def_emb = model.encode([d_text], convert_to_numpy=True)[0]
        denom = (np.linalg.norm(sent_emb) * np.linalg.norm(def_emb) + 1e-12)
        cos = float(np.dot(sent_emb, def_emb) / denom)
        scores.append(cos)

    pred_idx = int(np.argmax(scores))
    return pred_idx, scores

def predict_mixed_weighted_verbose(ex, model, q_system_mixed,
                                  context_window=10, chunk_chars=12, stride=6):
    sent = ex["sentence"]
    word = ex["target_word"]
    defs = ex["definitions"]

    sent_comps, sent_chunks = encode_context_components(
        sent, word, model,
        context_window=context_window,
        chunk_chars=chunk_chars,
        stride=stride
    )

    comp_norms = np.linalg.norm(sent_comps, axis=1) + 1e-12

    scores = []
    for d_text, _ in defs:
        def_emb = model.encode([d_text], convert_to_numpy=True)[0]
        def_norm = np.linalg.norm(def_emb) + 1e-12

        cos_vec = (sent_comps @ def_emb) / (comp_norms * def_norm)
        weights = np.maximum(0.0, cos_vec)
        if weights.sum() < 1e-12:
            weights = None

        p = q_system_mixed.measure_probability(sent_comps, def_emb, weights=weights)
        scores.append(float(p))

    pred_idx = int(np.argmax(scores))
    return pred_idx, scores, sent_chunks

def top2(scores):
    idxs = np.argsort(scores)[::-1]
    if len(idxs) == 0:
        return None, None
    best = int(idxs[0])
    second = int(idxs[1]) if len(idxs) > 1 else None
    return best, second

def show_disagreements(test_dataset, model, q_system_mixed,
                       max_show=10, search_limit=1000, seed=0,
                       context_window=10, chunk_chars=12, stride=6):
    rng = np.random.default_rng(seed)
    idxs = np.arange(len(test_dataset))
    rng.shuffle(idxs)

    shown = 0
    checked = 0

    for i in idxs:
        ex = test_dataset[i]
        checked += 1
        if checked > search_limit:
            break

        pred_c, sc_c = predict_cosine_verbose(ex, model)
        pred_m, sc_m, chunks = predict_mixed_weighted_verbose(
            ex, model, q_system_mixed,
            context_window=context_window,
            chunk_chars=chunk_chars,
            stride=stride
        )

        if pred_c == pred_m:
            continue  # 只看分歧

        gold = ex["gold_index"]
        defs = ex["definitions"]

        # 排名資訊
        c1, c2 = top2(sc_c)
        m1, m2 = top2(sc_m)

        print("=" * 80)
        print(f"[#{shown+1}] dataset_idx={i}")
        print(f"句子: {ex['sentence']}")
        print(f"目標詞: {ex['target_word']}")
        print("-" * 80)

        # gold
        gold_text = defs[gold][0]
        print(f"✅ GOLD [{gold}] {gold_text}")

        # cosine
        print("\n[Cosine]")
        print(f"pred = {pred_c} | score = {sc_c[pred_c]:.6f}")
        if c2 is not None:
            print(f"2nd  = {c2} | score = {sc_c[c2]:.6f} | margin = {(sc_c[c1]-sc_c[c2]):.6f}")

        # mixed
        print("\n[Word2State Mixed (weighted)]")
        print(f"pred = {pred_m} | prob  = {sc_m[pred_m]:.6f}")
        if m2 is not None:
            print(f"2nd  = {m2} | prob  = {sc_m[m2]:.6f} | margin = {(sc_m[m1]-sc_m[m2]):.6f}")

        # 顯示候選定義（簡短）
        print("\n候選定義（截短顯示）：")
        for k, (d_text, sid) in enumerate(defs):
            tag = []
            if k == gold: tag.append("GOLD")
            if k == pred_c: tag.append("C")
            if k == pred_m: tag.append("M")
            tag_str = ",".join(tag) if tag else ""
            print(f"  [{k}] {('['+tag_str+']').ljust(8)} {d_text[:60]}")

        print("\nMixed 使用的 chunks/components：")
        print(chunks)

        shown += 1
        if shown >= max_show:
            break

    print("\n" + "=" * 80)
    print(f"掃描題數: {min(checked, search_limit)} / {len(test_dataset)}")
    print(f"分歧題數（顯示）: {shown} / max_show={max_show}")

# ------------------ 執行：找 10 題分歧例子 ------------------
show_disagreements(
    test_dataset,
    bge_model,
    q_system_mixed,
    max_show=20,          # 想看更多就調大
    search_limit=10000,    # 想掃更久就調大
    seed=7,
    context_window=10,
    chunk_chars=8,
    stride=4
)


[#1] dataset_idx=19
句子: 月到中秋分外<明>。
目標詞: 明
--------------------------------------------------------------------------------
✅ GOLD [0] 形容光源的光線充足。

[Cosine]
pred = 0 | score = 0.462128
2nd  = 1 | score = 0.451255 | margin = 0.010873

[Word2State Mixed (weighted)]
pred = 1 | prob  = 0.138672
2nd  = 0 | prob  = 0.132428 | margin = 0.006244

候選定義（截短顯示）：
  [0] [GOLD,C] 形容光源的光線充足。
  [1] [M]      形容白天開始時太陽從地平線升起照亮天空。

Mixed 使用的 chunks/components：
['月到中秋分外<明']

掃描題數: 29 / 29
分歧題數（顯示）: 1 / max_show=20


In [ ]:
# ======================================================
# [新增 Cell] 找「mixed 贏 cosine」的 case study
# 條件：mixed 正確 & cosine 錯誤，且 chunks>=3、candidates>=4（可調）
# ======================================================

import numpy as np

def find_mixed_wins(
    test_dataset,
    model,
    q_system_mixed,
    max_show=10,
    search_limit=20000,
    seed=0,
    # mixed 的切片參數（跟你比較用的一致）
    context_window=10,
    chunk_chars=12,
    stride=6,
    # 篩選條件
    min_chunks=3,
    min_candidates=4
):
    rng = np.random.default_rng(seed)
    idxs = np.arange(len(test_dataset))
    rng.shuffle(idxs)

    shown = 0
    checked = 0
    eligible = 0

    for i in idxs:
        ex = test_dataset[i]
        checked += 1
        if checked > search_limit:
            break

        # 先用 mixed 算 chunks，順便做條件篩選
        pred_m, sc_m, chunks = predict_mixed_weighted_verbose(
            ex, model, q_system_mixed,
            context_window=context_window,
            chunk_chars=chunk_chars,
            stride=stride
        )

        if len(chunks) < min_chunks:
            continue
        if len(ex["definitions"]) < min_candidates:
            continue

        eligible += 1

        pred_c, sc_c = predict_cosine_verbose(ex, model)
        gold = ex["gold_index"]

        # 條件：mixed 對、cosine 錯
        if (pred_m == gold) and (pred_c != gold):
            defs = ex["definitions"]

            c1, c2 = top2(sc_c)
            m1, m2 = top2(sc_m)

            print("=" * 80)
            print(f"[#{shown+1}] dataset_idx={i} | eligible_checked={eligible}")
            print(f"句子: {ex['sentence']}")
            print(f"目標詞: {ex['target_word']}")
            print(f"chunks_count={len(chunks)} | candidates={len(defs)}")
            print("-" * 80)

            # gold
            print(f"✅ GOLD [{gold}] {defs[gold][0]}")

            # cosine
            print("\n[Cosine] (錯)")
            print(f"pred = {pred_c} | score = {sc_c[pred_c]:.6f}")
            if c2 is not None:
                print(f"2nd  = {c2} | score = {sc_c[c2]:.6f} | margin = {(sc_c[c1]-sc_c[c2]):.6f}")

            # mixed
            print("\n[Word2State Mixed (weighted)] (對)")
            print(f"pred = {pred_m} | prob  = {sc_m[pred_m]:.6f}")
            if m2 is not None:
                print(f"2nd  = {m2} | prob  = {sc_m[m2]:.6f} | margin = {(sc_m[m1]-sc_m[m2]):.6f}")

            # 候選定義（短顯示）
            print("\n候選定義（截短顯示）：")
            for k, (d_text, sid) in enumerate(defs):
                tag = []
                if k == gold: tag.append("GOLD")
                if k == pred_c: tag.append("C")
                if k == pred_m: tag.append("M")
                tag_str = ",".join(tag) if tag else ""
                print(f"  [{k}] {('['+tag_str+']').ljust(10)} {d_text[:70]}")

            print("\nMixed 使用的 chunks/components：")
            print(chunks)

            shown += 1
            if shown >= max_show:
                break

    print("\n" + "=" * 80)
    print(f"掃描題數: {min(checked, search_limit)} / {len(test_dataset)}")
    print(f"符合條件（chunks>={min_chunks}, candidates>={min_candidates}）題數: {eligible}")
    print(f"mixed 贏 cosine（顯示）: {shown} / max_show={max_show}")
    if shown == 0:
        print("⚠️ 沒找到 mixed 贏 cosine 的例子。建議：")
        print("  1) 把 search_limit 調大（例如 50000）")
        print("  2) 放寬條件 min_chunks/min_candidates")
        print("  3) 換 seed")
        print("  4) 調 chunk_chars/stride（更細或更粗）")

# ------------------ 執行：找 10 題 mixed 贏 cosine ------------------
find_mixed_wins(
    test_dataset,
    bge_model,
    q_system_mixed,
    max_show=10,
    search_limit=50000,
    seed=123,
    context_window=10,
    chunk_chars=16,
    stride=8,
    min_chunks=2,
    min_candidates=2
)



掃描題數: 29 / 29
符合條件（chunks>=2, candidates>=2）題數: 0
mixed 贏 cosine（顯示）: 0 / max_show=10
⚠️ 沒找到 mixed 贏 cosine 的例子。建議：
  1) 把 search_limit 調大（例如 50000）
  2) 放寬條件 min_chunks/min_candidates
  3) 換 seed
  4) 調 chunk_chars/stride（更細或更粗）


In [ ]:

'''import numpy as np
import random
from collections import defaultdict
from tqdm.auto import tqdm  # 自動選擇適合 Colab 的進度條

# ==========================================
# 1. 確保輔助函數存在 (防止前面漏跑)
# ==========================================

def encode_context_components_safe(text, target_word, model, context_window=500, chunk_chars=8, stride=4):
    """將上下文切片 (用於混合態)"""
    idx = text.find(target_word)
    if idx != -1:
        start = max(0, idx - context_window)
        end = min(len(text), idx + len(target_word) + context_window)
        context = text[start:end]
    else:
        context = text

    chunks = []
    for s in range(0, max(1, len(context) - chunk_chars + 1), stride):
        chunk = context[s:s + chunk_chars].strip()
        if chunk:
            chunks.append(chunk)
    if not chunks: chunks = [context]

    # model 依賴全域變數 bge_model
    comps = model.encode(chunks, convert_to_numpy=True)
    return comps, chunks

# ==========================================
# 2. 定義資料集增強函數 (含進度條)
# ==========================================

def build_enhanced_dataset(ds_split, max_instances=1000, min_candidates=4):
    # A. 建立詞義倉庫 (偽字典)
    print("📚 正在建立詞義倉庫 (Sense Inventory)...")
    sense_inventory = defaultdict(set)
    for ex in ds_split:
        if ex['cwn_definition'] and ex['cwn_sense_id']:
            sense_inventory[ex['test_word']].add((ex['cwn_definition'], ex['cwn_sense_id']))

    # B. 分組原始資料
    groups = defaultdict(list)
    for ex in ds_split:
        groups[(ex["test_word"], ex["test_sentence"])].append(ex)

    test_dataset = []

    # C. 製作增強版資料集
    print(f"🚀 開始建立增強版測試集 (目標: {max_instances} 題)...")
    pbar = tqdm(total=max_instances, desc="Building Dataset")

    # 隨機打亂處理順序，避免只取到同一類的詞
    all_keys = list(groups.keys())
    random.shuffle(all_keys)

    for key in all_keys:
        if len(test_dataset) >= max_instances: break

        word, sent = key
        rows = groups[key]

        # 收集現有選項
        definitions = []
        seen_sids = set()
        gold_index = None

        for r in rows:
            sid = r['cwn_sense_id']
            defi = r['cwn_definition']
            label = r['label']
            if sid not in seen_sids:
                definitions.append((defi, sid))
                seen_sids.add(sid)
                if label: gold_index = len(definitions) - 1

        if gold_index is None: continue # 沒答案跳過

        # 補足選項 (Distractors)
        if len(definitions) < min_candidates:
            all_senses = list(sense_inventory.get(word, []))
            # 排除已存在的正確答案
            distractors = [s for s in all_senses if s[1] not in seen_sids]

            needed = min_candidates - len(definitions)
            # 如果不夠補，就全補；夠補就隨機選
            if len(distractors) >= needed:
                selected = random.sample(distractors, needed)
            else:
                selected = distractors
            definitions.extend(selected)

        # 補完還是少於2個選項，跳過
        if len(definitions) < 2: continue

        # 重新打亂順序並更新答案位置
        gold_sid = definitions[gold_index][1]
        random.shuffle(definitions)
        new_gold_index = -1
        for i, (d, s) in enumerate(definitions):
            if s == gold_sid:
                new_gold_index = i
                break

        test_dataset.append({
            "sentence": sent,
            "target_word": word,
            "definitions": definitions,
            "gold_index": new_gold_index
        })
        pbar.update(1)

    pbar.close()
    return test_dataset

# ==========================================
# 3. 定義評估流程 (含進度條)
# ==========================================

def evaluate_all_methods(dataset, model, q_pure, q_mixed):
    results = {"Cosine": 0, "Quantum(Pure)": 0, "Word2State(Mixed)": 0}

    print("\n⚡ 開始評估三種模型效能...")
    for ex in tqdm(dataset, desc="Evaluating", unit="q"):
        sent = ex["sentence"]
        word = ex["target_word"]
        defs = ex["definitions"]
        gold = ex["gold_index"]

        # 1. Cosine Baseline
        # --------------------------------
        # 這裡直接實作簡單版 Cosine 邏輯
        # 為了避免前面函數沒定義，這裡直接寫
        sent_emb = model.encode([sent], convert_to_numpy=True)[0] # 簡單起見用整句
        # 若要精確可用: sent_emb = encode_target_context(sent, word, model)[0]

        cos_scores = []
        for d_text, _ in defs:
            def_emb = model.encode([d_text], convert_to_numpy=True)[0]
            cos = np.dot(sent_emb, def_emb) / (np.linalg.norm(sent_emb)*np.linalg.norm(def_emb)+1e-9)
            cos_scores.append(cos)
        if np.argmax(cos_scores) == gold: results["Cosine"] += 1

        # 2. Quantum Pure (假設 q_pure 變數存在)
        # --------------------------------
        # 這裡需要用到前面定義的 encode_target_context
        # 如果你沒跑前面 Cell，這行可能會報錯，請確保 Step 5 有執行
        try:
            # 嘗試使用 Context Window Embedding
            sent_context_emb = encode_target_context(sent, word, model)[0]
        except:
            # fallback
            sent_context_emb = sent_emb

        pure_scores = []
        for d_text, _ in defs:
            def_emb = model.encode([d_text], convert_to_numpy=True)[0]
            p = q_pure.measure_similarity(sent_context_emb, def_emb)
            pure_scores.append(p)
        if np.argmax(pure_scores) == gold: results["Quantum(Pure)"] += 1

        # 3. Word2State Mixed
        # --------------------------------
        sent_comps, _ = encode_context_components_safe(
            sent, word, model, context_window=10, chunk_chars=12, stride=6
        )
        comp_norms = np.linalg.norm(sent_comps, axis=1) + 1e-12

        mixed_scores = []
        for d_text, _ in defs:
            def_emb = model.encode([d_text], convert_to_numpy=True)[0]
            def_norm = np.linalg.norm(def_emb) + 1e-12

            # 使用定義相關權重
            cos_vec = (sent_comps @ def_emb) / (comp_norms * def_norm)
            weights = np.maximum(0.0, cos_vec)
            if weights.sum() < 1e-12: weights = None

            p = q_mixed.measure_probability(sent_comps, def_emb, weights=weights)
            mixed_scores.append(p)
        if np.argmax(mixed_scores) == gold: results["Word2State(Mixed)"] += 1

    # 統計結果
    n = len(dataset)
    print(f"\n📊 最終評估報告 (N={n}):")
    print("="*40)
    for name, correct in results.items():
        acc = correct / n
        print(f"{name:<20} | 準確率: {acc:.2%}")
    print("="*40)

# ==========================================
# 4. 執行主程式
# ==========================================

# 1. 建立擴充資料集 (目標 1000 筆)
# ds 和 bge_model 來自前面的 Cell，q_system 和 q_system_mixed 也是
enhanced_test_set = build_enhanced_dataset(ds['train'], max_instances=300, min_candidates=4)

# 2. 執行評估
evaluate_all_methods(enhanced_test_set, bge_model, q_system, q_system_mixed)
'''

'import numpy as np\nimport random\nfrom collections import defaultdict\nfrom tqdm.auto import tqdm  # 自動選擇適合 Colab 的進度條\n\n# ==========================================\n# 1. 確保輔助函數存在 (防止前面漏跑)\n# ==========================================\n\ndef encode_context_components_safe(text, target_word, model, context_window=500, chunk_chars=8, stride=4):\n    """將上下文切片 (用於混合態)"""\n    idx = text.find(target_word)\n    if idx != -1:\n        start = max(0, idx - context_window)\n        end = min(len(text), idx + len(target_word) + context_window)\n        context = text[start:end]\n    else:\n        context = text\n\n    chunks = []\n    for s in range(0, max(1, len(context) - chunk_chars + 1), stride):\n        chunk = context[s:s + chunk_chars].strip()\n        if chunk:\n            chunks.append(chunk)\n    if not chunks: chunks = [context]\n\n    # model 依賴全域變數 bge_model\n    comps = model.encode(chunks, convert_to_numpy=True)\n    return comps, chunks\n\n# ==============================

In [ ]:
'''import numpy as np
import random
from collections import defaultdict
# 嘗試使用 tqdm，如果環境不支援則退回普通迭代器
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs): return iterable

# ==========================================
# 1. 確保輔助函數存在 (安全檢查)
# ==========================================
# 確保這些核心函數已定義，如果前面 Cell 沒跑，這裡會報錯提醒
assert 'encode_context_components' in globals(), "❌ 請先執行定義 'encode_context_components' 的 Cell"
assert 'predict_cosine' in globals(), "❌ 請先執行定義 'predict_cosine' 的 Cell"
assert 'predict_quantum_pure' in globals(), "❌ 請先執行定義 'predict_quantum_pure' 的 Cell"
assert 'predict_word2state_mixed_weighted' in globals(), "❌ 請先執行定義 'predict_word2state_mixed_weighted' 的 Cell"

# ==========================================
# 2. 定義帶長度篩選的資料集建構函數
# ==========================================
def build_filtered_test_dataset(
    ds_split,
    sense_inventory,  # 這是前面步驟建立的詞義倉庫
    max_instances=100,
    min_candidates=4,
    min_length=0      # ✨ 新增：最小句子長度篩選
):
    # A. 分組
    print(f"🔍 正在篩選資料 (最小長度={min_length})...")
    groups = defaultdict(list)
    for ex in ds_split:
        # ✨ 在這裡直接過濾短句子，節省記憶體與處理時間
        if len(ex["test_sentence"]) >= min_length:
            groups[(ex["test_word"], ex["test_sentence"])].append(ex)

    test_dataset = []

    # B. 製作增強版資料集
    print(f"🚀 開始建立測試集 (目標: {max_instances} 題, 符合長度條件的候選句群: {len(groups)} 組)...")
    pbar = tqdm(total=max_instances, desc="Building Dataset")

    # 隨機打亂處理順序
    all_keys = list(groups.keys())
    random.shuffle(all_keys)

    for key in all_keys:
        if len(test_dataset) >= max_instances: break

        word, sent = key
        rows = groups[key]

        # 收集現有選項
        definitions = []
        seen_sids = set()
        gold_index = None

        for r in rows:
            sid = r['cwn_sense_id']
            defi = r['cwn_definition']
            label = r['label']
            if sid not in seen_sids:
                definitions.append((defi, sid))
                seen_sids.add(sid)
                if label: gold_index = len(definitions) - 1

        if gold_index is None: continue

        # 補足選項 (Distractors)
        if len(definitions) < min_candidates:
            all_senses = list(sense_inventory.get(word, []))
            distractors = [s for s in all_senses if s[1] not in seen_sids]
            needed = min_candidates - len(definitions)
            if len(distractors) >= needed:
                selected = random.sample(distractors, needed)
            else:
                selected = distractors
            definitions.extend(selected)

        if len(definitions) < 2: continue

        # 重新打亂順序並更新答案位置
        gold_sid = definitions[gold_index][1]
        random.shuffle(definitions)
        new_gold_index = -1
        for i, (d, s) in enumerate(definitions):
            if s == gold_sid:
                new_gold_index = i
                break
        if new_gold_index == -1:
            continue
        test_dataset.append({
            "sentence": sent,
            "target_word": word,
            "definitions": definitions,
            "gold_index": new_gold_index
        })
        pbar.update(1)

    pbar.close()
    return test_dataset

# ==========================================
# 3. 執行評估流程
# ==========================================

# A. 確保詞義倉庫已建立 (如果之前跑過可以註解掉，但重跑無害且很快)
if 'sense_inventory' not in globals() or not sense_inventory:
    print("📚 重建詞義倉庫 (Sense Inventory)...")
    sense_inventory = defaultdict(set)
    for ex in ds['train']:
        if ex['cwn_definition'] and ex['cwn_sense_id']:
            sense_inventory[ex['test_word']].add((ex['cwn_definition'], ex['cwn_sense_id']))

# B. 建立長難句測試集 (長度 >= 30)
# 您可以調整 min_length 來觀察不同長度對混合態優勢的影響
# 例如: 10 (一般), 30 (長句), 50 (特長句)
min_len = 30
long_test_dataset = build_filtered_test_dataset(
    ds['train'],
    sense_inventory,
    max_instances=100,
    min_candidates=4,
    min_length=min_len
)

# C. 執行評估
if len(long_test_dataset) > 0:
    print(f"\n⚡ 開始評估長難句效能 (長度 >= {min_len})...")

    # 這裡重複定義 evaluate 函數是為了確保它能用到全域變數，且讓這段代碼自成一體
    def run_evaluation(dataset, model, q_pure, q_mixed):
        results = {"Cosine": 0, "Quantum(Pure)": 0, "Word2State(Mixed)": 0}
        for ex in tqdm(dataset, desc="Evaluating"):
            gold = ex["gold_index"]

            pred_c, _ = predict_cosine(ex, model)
            if pred_c == gold: results["Cosine"] += 1

            pred_p, _ = predict_quantum_pure(ex, model, q_pure)
            if pred_p == gold: results["Quantum(Pure)"] += 1

            pred_m, _ = predict_word2state_mixed_weighted(ex, model, q_mixed)
            if pred_m == gold: results["Word2State(Mixed)"] += 1

        n = len(dataset)
        return {k: v/n for k, v in results.items()}

    final_results = run_evaluation(
        long_test_dataset,
        bge_model,
        q_system,
        q_system_mixed
    )

    print(f"\n📊 長難句評估報告 (N={len(long_test_dataset)}):")
    print("="*40)
    for name, acc in final_results.items():
        print(f"{name:<20} | 準確率: {acc:.2%}")
    print("="*40)

    # 簡單分析混合態是否有優勢
    mixed_acc = final_results["Word2State(Mixed)"]
    cosine_acc = final_results["Cosine"]
    if mixed_acc > cosine_acc:
        print(f"🚀 混合態勝出！提升幅度: +{(mixed_acc - cosine_acc)*100:.2f}%")
    else:
        print(f"⚖️ 混合態與 Cosine 表現相當或略低 (差異: {(mixed_acc - cosine_acc)*100:.2f}%)")

else:
    print("⚠️ 警告：沒有找到符合長度條件的句子，請嘗試降低 min_length。")'''

'import numpy as np\nimport random\nfrom collections import defaultdict\n# 嘗試使用 tqdm，如果環境不支援則退回普通迭代器\ntry:\n    from tqdm.auto import tqdm\nexcept ImportError:\n    def tqdm(iterable, **kwargs): return iterable\n\n# ==========================================\n# 1. 確保輔助函數存在 (安全檢查)\n# ==========================================\n# 確保這些核心函數已定義，如果前面 Cell 沒跑，這裡會報錯提醒\nassert \'encode_context_components\' in globals(), "❌ 請先執行定義 \'encode_context_components\' 的 Cell"\nassert \'predict_cosine\' in globals(), "❌ 請先執行定義 \'predict_cosine\' 的 Cell"\nassert \'predict_quantum_pure\' in globals(), "❌ 請先執行定義 \'predict_quantum_pure\' 的 Cell"\nassert \'predict_word2state_mixed_weighted\' in globals(), "❌ 請先執行定義 \'predict_word2state_mixed_weighted\' 的 Cell"\n\n# ==========================================\n# 2. 定義帶長度篩選的資料集建構函數\n# ==========================================\ndef build_filtered_test_dataset(\n    ds_split,\n    sense_inventory,  # 這是前面步驟建立的詞義倉庫\n    max_instances=100,\n    min_candidates=4,\n 

In [ ]:
import numpy as np
import pandas as pd
import random
from collections import defaultdict
from tqdm.auto import tqdm

# =========================================================
# A) 建 Sense Inventory（若你已經有 sense_inventory，可跳過）
# =========================================================
def build_sense_inventory(ds_split):
    inv = defaultdict(set)
    for ex in ds_split:
        if ex.get("cwn_definition") and ex.get("cwn_sense_id"):
            inv[ex["test_word"]].add((ex["cwn_definition"], ex["cwn_sense_id"]))
    return inv

if "sense_inventory" not in globals() or not sense_inventory:
    print("📚 正在建立詞義倉庫 (Sense Inventory)...")
    sense_inventory = build_sense_inventory(ds["train"])

# =========================================================
# B) 建 dataset：同一套 builder，支援 min/max 長度
# =========================================================
def build_test_dataset_length_filtered(
    ds_split,
    sense_inventory,
    max_instances=100,
    min_candidates=4,
    min_length=0,
    max_length=None,
    seed=42
):
    rng = random.Random(seed)

    # 分組：同詞同句聚在一起
    groups = defaultdict(list)
    for ex in ds_split:
        sent = ex["test_sentence"]
        L = len(sent)
        if L < min_length:
            continue
        if (max_length is not None) and (L > max_length):
            continue
        groups[(ex["test_word"], sent)].append(ex)

    print(f"🚀 建立測試集 (目標: {max_instances}, min_len={min_length}, max_len={max_length}, groups={len(groups)})")
    keys = list(groups.keys())
    rng.shuffle(keys)

    test_dataset = []
    pbar = tqdm(total=max_instances, desc="Building Dataset")

    for key in keys:
        if len(test_dataset) >= max_instances:
            break

        word, sent = key
        rows = groups[key]

        # 先收集該句已出現的 senses（含 gold）
        definitions = []
        seen_sids = set()
        gold_index = None

        for r in rows:
            sid = r["cwn_sense_id"]
            defi = r["cwn_definition"]
            label = r["label"]
            if sid not in seen_sids:
                definitions.append((defi, sid))
                seen_sids.add(sid)
                if label:
                    gold_index = len(definitions) - 1

        if gold_index is None:
            continue

        # 補 distractors（同一個詞的其他 sense）
        if len(definitions) < min_candidates:
            all_senses = list(sense_inventory.get(word, []))
            distractors = [s for s in all_senses if s[1] not in seen_sids]
            need = min_candidates - len(definitions)
            if len(distractors) >= need:
                definitions.extend(rng.sample(distractors, need))
            else:
                definitions.extend(distractors)

        if len(definitions) < 2:
            continue

        # shuffle 並更新 gold_index
        gold_sid = definitions[gold_index][1]
        rng.shuffle(definitions)

        new_gold_index = -1
        for i, (_, sid) in enumerate(definitions):
            if sid == gold_sid:
                new_gold_index = i
                break
        if new_gold_index == -1:
            continue

        test_dataset.append({
            "sentence": sent,
            "target_word": word,
            "definitions": definitions,   # [(def_text, sid), ...]
            "gold_index": new_gold_index
        })
        pbar.update(1)

    pbar.close()
    return test_dataset

# =========================================================
# C) 分析工具：統一三模型指標
# =========================================================
def _unpack(out):
    # 支援 (pred_idx, scores) 或 (pred_idx, scores, extra)
    if isinstance(out, tuple) and len(out) >= 2:
        return int(out[0]), np.array(out[1], dtype=float)
    raise ValueError("predict_* 必須至少回傳 (pred_idx, scores)")

def _margin(scores):
    if len(scores) < 2:
        return 0.0
    s = np.sort(scores)[::-1]
    return float(s[0] - s[1])

def _gold_rank(scores, gold_idx):
    order = np.argsort(scores)[::-1]
    pos = int(np.where(order == gold_idx)[0][0])
    return pos + 1  # 1-based

def _topk_hit(scores, gold_idx, k):
    order = np.argsort(scores)[::-1]
    return int(gold_idx in order[:k])

def evaluate_and_collect(dataset, model, q_pure, q_mixed, desc=""):
    rows = []

    for ex in tqdm(dataset, desc=f"Evaluating {desc}", unit="q"):
        gold = ex["gold_index"]
        defs = ex["definitions"]
        if not (0 <= gold < len(defs)):
            continue

        # 取得三模型輸出
        pred_c, sc_c = _unpack(predict_cosine(ex, model))
        pred_p, sc_p = _unpack(predict_quantum_pure(ex, model, q_pure))
        pred_m, sc_m = _unpack(predict_word2state_mixed_weighted(ex, model, q_mixed))

        # NaN 防護
        sc_c = np.nan_to_num(sc_c, nan=-1e9, posinf=1e9, neginf=-1e9)
        sc_p = np.nan_to_num(sc_p, nan=-1e9, posinf=1e9, neginf=-1e9)
        sc_m = np.nan_to_num(sc_m, nan=-1e9, posinf=1e9, neginf=-1e9)

        gold_def, gold_sid = defs[gold]

        def pack(prefix, pred, scores):
            return {
                f"{prefix}_pred": pred,
                f"{prefix}_correct": int(pred == gold),
                f"{prefix}_top1": float(np.max(scores)),
                f"{prefix}_margin": _margin(scores),
                f"{prefix}_top2": _topk_hit(scores, gold, 2),
                f"{prefix}_top3": _topk_hit(scores, gold, 3),
                f"{prefix}_rank": _gold_rank(scores, gold),
                f"{prefix}_rr": 1.0 / _gold_rank(scores, gold)
            }

        row = {
            "length": len(ex["sentence"]),
            "K": len(defs),
            "target_word": ex["target_word"],
            "sentence": ex["sentence"],
            "gold_index": gold,
            "gold_sid": gold_sid,
            "gold_def": gold_def
        }
        row.update(pack("cos", pred_c, sc_c))
        row.update(pack("pure", pred_p, sc_p))
        row.update(pack("mixed", pred_m, sc_m))

        # 存 pred 定義（方便錯誤案例）
        row["cos_pred_def"] = defs[pred_c][0]
        row["pure_pred_def"] = defs[pred_p][0]
        row["mixed_pred_def"] = defs[pred_m][0]

        rows.append(row)

    return pd.DataFrame(rows)

def summarize(df):
    def one(prefix):
        return pd.Series({
            "accuracy": df[f"{prefix}_correct"].mean(),
            "top2_acc": df[f"{prefix}_top2"].mean(),
            "top3_acc": df[f"{prefix}_top3"].mean(),
            "MRR": df[f"{prefix}_rr"].mean(),
            "avg_margin": df[f"{prefix}_margin"].mean(),
            "avg_K": df["K"].mean(),
            "avg_len": df["length"].mean()
        })
    out = pd.DataFrame({
        "Cosine": one("cos"),
        "Quantum(Pure)": one("pure"),
        "Word2State(Mixed)": one("mixed")
    }).T
    return out

def top_confident_errors(df, prefix, topn=10):
    d = df[df[f"{prefix}_correct"] == 0].copy()
    d = d.sort_values([f"{prefix}_margin", f"{prefix}_top1"], ascending=False)
    cols = ["target_word","length","K","sentence","gold_def",
            f"{prefix}_top1", f"{prefix}_margin"]
    pred_def_col = f"{prefix}_pred_def"
    return d[cols + [pred_def_col]].head(topn)

# =========================================================
# D) 一鍵：短句100 + 長句100，並輸出分析
# =========================================================
def run_short_long_analysis(
    ds_split,
    sense_inventory,
    model,
    q_pure,
    q_mixed,
    n_short=100,
    n_long=100,
    short_max_len=15,
    long_min_len=30,
    min_candidates=4,
    seed=42
):
    # --- 建短句集 ---
    short_set = build_test_dataset_length_filtered(
        ds_split, sense_inventory,
        max_instances=n_short,
        min_candidates=min_candidates,
        min_length=0,
        max_length=short_max_len,
        seed=seed
    )

    # --- 建長句集 ---
    long_set = build_test_dataset_length_filtered(
        ds_split, sense_inventory,
        max_instances=n_long,
        min_candidates=min_candidates,
        min_length=long_min_len,
        max_length=None,
        seed=seed+1
    )

    print(f"\n✅ 短句集 N={len(short_set)}（len <= {short_max_len}）")
    print(f"✅ 長句集 N={len(long_set)}（len >= {long_min_len}）")

    # --- 評估短句 ---
    df_short = evaluate_and_collect(short_set, model, q_pure, q_mixed, desc="SHORT")
    print("\n=== SHORT Overall Summary ===")
    print(summarize(df_short))

    print("\n--- SHORT Top Confident Errors (Cosine) ---")
    print(top_confident_errors(df_short, "cos", 10))
    print("\n--- SHORT Top Confident Errors (Pure) ---")
    print(top_confident_errors(df_short, "pure", 10))
    print("\n--- SHORT Top Confident Errors (Mixed) ---")
    print(top_confident_errors(df_short, "mixed", 10))

    # --- 評估長句 ---
    df_long = evaluate_and_collect(long_set, model, q_pure, q_mixed, desc="LONG")
    print("\n=== LONG Overall Summary ===")
    print(summarize(df_long))

    print("\n--- LONG Top Confident Errors (Cosine) ---")
    print(top_confident_errors(df_long, "cos", 10))
    print("\n--- LONG Top Confident Errors (Pure) ---")
    print(top_confident_errors(df_long, "pure", 10))
    print("\n--- LONG Top Confident Errors (Mixed) ---")
    print(top_confident_errors(df_long, "mixed", 10))

    return short_set, long_set, df_short, df_long

# ================== 你只要執行這行 ==================
short_set, long_set, df_short, df_long = run_short_long_analysis(
    ds_split=ds["train"],
    sense_inventory=sense_inventory,
    model=bge_model,
    q_pure=q_system,
    q_mixed=q_system_mixed,
    n_short=100,
    n_long=100,
    short_max_len=15,   # 你可改：短句上限
    long_min_len=30,    # 你可改：長句下限
    min_candidates=4,
    seed=42
)

📚 正在建立詞義倉庫 (Sense Inventory)...
🚀 建立測試集 (目標: 100, min_len=0, max_len=15, groups=11480)


Building Dataset:   0%|          | 0/100 [00:00<?, ?it/s]

🚀 建立測試集 (目標: 100, min_len=30, max_len=None, groups=1276)


Building Dataset:   0%|          | 0/100 [00:00<?, ?it/s]


✅ 短句集 N=100（len <= 15）
✅ 長句集 N=100（len >= 30）


Evaluating SHORT:   0%|          | 0/100 [00:00<?, ?q/s]


=== SHORT Overall Summary ===
                   accuracy  top2_acc  top3_acc       MRR  avg_margin  avg_K  \
Cosine                 0.32      0.51      0.64  0.517761    0.038837  13.75   
Quantum(Pure)          0.32      0.51      0.64  0.517761    0.040904  13.75   
Word2State(Mixed)      0.34      0.51      0.64  0.526094    0.039718  13.75   

                   avg_len  
Cosine               11.06  
Quantum(Pure)        11.06  
Word2State(Mixed)    11.06  

--- SHORT Top Confident Errors (Cosine) ---
   target_word  length   K         sentence              gold_def  cos_top1  \
26           掛      14  18   一隻黑色皮袋則<掛>在右肩，   支撐特定物體的一端使其維持在特定高度。  0.613250   
47           用      14   4   我們賺這麼多錢有什麼<用>？          特定對象所能發生的效能。  0.502165   
89           就       8  19         <就>讓它過去，   表後述命題和前文相對，具有轉折的語氣。  0.656369   
33           面      12   5     <面>無表情地望著大家，         動物頭從額頭到下巴的部分。  0.589800   
39           字      15   8  唱歌對我來說只有一個唱<字>，          中文書面書寫的最小單位。  0.609459   
0            

Evaluating LONG:   0%|          | 0/100 [00:00<?, ?q/s]


=== LONG Overall Summary ===
                   accuracy  top2_acc  top3_acc       MRR  avg_margin  avg_K  \
Cosine                 0.25      0.46      0.62  0.477041    0.031143  12.15   
Quantum(Pure)          0.25      0.46      0.62  0.477041    0.029305  12.15   
Word2State(Mixed)      0.30      0.48      0.62  0.511064    0.029840  12.15   

                   avg_len  
Cosine               38.37  
Quantum(Pure)        38.37  
Word2State(Mixed)    38.37  

--- LONG Top Confident Errors (Cosine) ---
   target_word  length   K                                           sentence  \
26           心      31  12                    打造「總裁獅子<心>」、「乞丐囝仔」超級暢銷書旋風的幕後推手，   
36           上      70  17  這個工具尤其在ｓｈｅｌｌ－ｃｏｍｍａｎｄ、ｇｒｅｐ、ｏｃｃｕｒ、ｑｕｅｒｙ－ｒｅｐｌａｃｅ和...   
45           明      55   4  台北社區‧文山區山胞豐年祭<明>舉行‧台北訊‧文山區本年度山胞豐年祭於八月廿九日（星期日）上...   
6            去      30  14                     馬來西亞的Ｈｕｓｓｅｉｎ和我一齊<去>Ｂｏｓｔｏｎ街上觀光。   
81           後      32   4                   農業部分凡公司投資計畫完成年度及其前、<後>一年度的三年期間內，   
43 

In [ ]:
[c for c in ["bge_model","model","embed_model","encoder","sentence_model","sbert_model"] if c in globals()]

['bge_model']